# Шаг 5. Загрузка и консолидация данных

In [2]:
# Импортируем библиотеку для работы с табличными данными
import pandas as pd

# Загружаем первый датасет: исторические периоды наборов
df_periods = pd.read_csv('review_periods.csv')

# Загружаем второй датасет: национальности в наборах (широкий формат)
df_nat_wide = pd.read_csv('reviews_by_nationality_wide.csv')

# Загружаем третий датасет: метаданные наборов с сайта PSR
df_psr = pd.read_csv('psr_enriched_data.csv')

# Проверяем, что все три файла загружены корректно
print("=== Проверка загрузки данных ===")
print(f"df_periods: {df_periods.shape[0]} строк, {df_periods.shape[1]} колонок")
print(f"df_nat_wide: {df_nat_wide.shape[0]} строк, {df_nat_wide.shape[1]} колонок")
print(f"df_psr: {df_psr.shape[0]} строк, {df_psr.shape[1]} колонок")

=== Проверка загрузки данных ===
df_periods: 2840 строк, 4 колонок
df_nat_wide: 2476 строк, 13 колонок
df_psr: 2840 строк, 5 колонок


## План консолидации
1. **Загрузка и проверка ключей объединения** — поле ID (единый для всех трёх таблиц).
**Базовая таблица** — review_periods.csv (2837 строк, все наборы).
1. **Присоединение к базовой таблице psr_enriched_data.csv** — left join по ID (все 2837 строк сохраняются).
2. **Перевод таблицы reviews_by_nationality_wide.csv из широкого формата в длинный** (melt).
3. **Присоединение таблицы reviews_by_nationality_wide.csv к базовой таблице** - left join. Наборы без национальностей (361 шт.) остаются одной строкой со значением NaN.
**Два уровня наблюдения:**
- df_sets — одна строка = один набор (для анализа производителей, годов выпуска, рейтингов);
- df_long — одна строка = пара «набор — национальность» (для анализа покрытия ниш и карты периодов и национальностей).

## Шаг 5.1. Загрузка и проверка ключей

In [3]:
import pandas as pd

# Загружаем три датасета
df_periods = pd.read_csv('review_periods.csv')
df_nat_wide = pd.read_csv('reviews_by_nationality_wide.csv')
df_psr = pd.read_csv('psr_enriched_data.csv')

# Проверяем, что ID уникален в каждой таблице (иначе join исказит строки)
print("ID уникален в periods:", df_periods['ID'].is_unique)
print("ID уникален в psr:", df_psr['ID'].is_unique)
print("ID уникален в nat_wide:", df_nat_wide['ID'].is_unique)

# Проверяем, что все ID из таблиц-спутников есть в базовой таблице
print("nat_wide ⊆ periods:", set(df_nat_wide['ID']).issubset(set(df_periods['ID'])))
print("psr ⊆ periods:", set(df_psr['ID']).issubset(set(df_periods['ID'])))

ID уникален в periods: True
ID уникален в psr: True
ID уникален в nat_wide: True
nat_wide ⊆ periods: False
psr ⊆ periods: True


Проверка ключей выявила несоответствие: в таблице национальностей есть ID, которых нет в таблице периодов.
Проведём диагностику расхождения.

## Шаг 5.1.1. Диагностика расхождения

In [4]:
# ID, которые есть в национальностях, но отсутствуют в периодах
extra_ids = sorted(set(df_nat_wide['ID']) - set(df_periods['ID']))
print("Количество 'лишних' ID в nat_wide:", len(extra_ids))

# Смотрим сами строки, чтобы понять их природу
df_nat_wide[df_nat_wide['ID'].isin(extra_ids)]

Количество 'лишних' ID в nat_wide: 3


,ID,Header,Nationality_1,Nationality_2,Nationality_3,Nationality_4,Nationality_5,Nationality_6,Nationality_7,Nationality_8,Nationality_9,Nationality_10,Nationality_11
867,1079,HaT Sumerian Infantry (8132),Iraqi,Sumerian,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
868,1080,HaT Sumerian Chariots (8130),Iraqi,Sumerian,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2401,2987,Linear-A Alexander the Great with General Staf...,Macedonian,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Выявлено, что три набора присутствовали в таблице национальностей, но отсутствовали в результатах парсинга периодов.
Значения Release_Year, Aggregate_Rating, Num_Figures взяты со страниц обзоров сайта (id=1079, 1080, 2987) 20.08.2026.

Исторические периоды определены методом экспертной оценки.
Принято:
- для ID 1079 и ID 1080 (шумеры) временной период с 2024 г. до н.э. по 2000 г. до н.э. (цивилизация существовала и ранее, но период условно ограничен выборкой датасета);
- для ID 2987 (Александр Македонский) период принят c 340 г. до н.э. по 323 г. до н.э.

## Шаг 5.1.2. Корректировка датасета

In [6]:
headers = df_nat_wide.set_index('ID')['Header']

manual_periods = pd.DataFrame([
    {'ID': 1079, 'Header': headers.loc[1079], 'Years_from': -2024, 'Years_to': -2000},
    {'ID': 1080, 'Header': headers.loc[1080], 'Years_from': -2024, 'Years_to': -2000},
    {'ID': 2987, 'Header': headers.loc[2987], 'Years_from': -340,  'Years_to': -323},
])

manual_psr = pd.DataFrame([
    {'ID': 1079, 'Header': headers.loc[1079], 'Release_Year': 2006, 'Aggregate_Rating': 40, 'Num_Figures': 92},
    {'ID': 1080, 'Header': headers.loc[1080], 'Release_Year': 2006, 'Aggregate_Rating': 42, 'Num_Figures': 6},
    {'ID': 2987, 'Header': headers.loc[2987], 'Release_Year': 2026, 'Aggregate_Rating': 48, 'Num_Figures': 5},
])

df_periods = pd.concat([df_periods, manual_periods], ignore_index=True)
df_psr     = pd.concat([df_psr,     manual_psr],     ignore_index=True)

df_periods.to_csv('review_periods.csv', index=False, encoding='utf-8')
df_psr.to_csv('psr_enriched_data.csv', index=False, encoding='utf-8')

print("periods:", len(df_periods), "| psr:", len(df_psr))

periods: 2840 | psr: 2840


## Шаг 5.1. Загрузка и проверка ключей (повторно, с учётом корректировки)

In [3]:
import pandas as pd

# Загружаем три датасета
df_periods = pd.read_csv('review_periods.csv')
df_nat_wide = pd.read_csv('reviews_by_nationality_wide.csv')
df_psr = pd.read_csv('psr_enriched_data.csv')

# Проверяем, что ID уникален в каждой таблице (иначе join исказит строки)
print("ID уникален в periods:", df_periods['ID'].is_unique)
print("ID уникален в psr:", df_psr['ID'].is_unique)
print("ID уникален в nat_wide:", df_nat_wide['ID'].is_unique)

# Проверяем, что все ID из таблиц-спутников есть в базовой таблице
print("nat_wide ⊆ periods:", set(df_nat_wide['ID']).issubset(set(df_periods['ID'])))
print("psr ⊆ periods:", set(df_psr['ID']).issubset(set(df_periods['ID'])))

ID уникален в periods: True
ID уникален в psr: True
ID уникален в nat_wide: True
nat_wide ⊆ periods: True
psr ⊆ periods: True


## Шаг 5.2. Объединение с метаданными сайта и извлечение информации о производителе

In [5]:
# Присоединяем метаданные PSR (Header дублируется, поэтому убираем его из правой таблицы)
df = df_periods.merge(df_psr.drop(columns=['Header']), on='ID', how='left')

# Справочник производителей взят со страницы Manufacturers сайта PSR.
# Сортируем по длине по убыванию, чтобы "A Call To Arms" проверялось раньше, чем "A".
MANUFACTURERS = [
    'A Call To Arms', 'Accurate', 'Airfix', 'Almark', 'Armourfast', 'Atlantic',
    'Billy V', 'BUM', 'Caesar', 'Co.Ma.', 'Coates & Shine', 'Dapol',
    'Dark Dream Studio', 'Eagle Games', 'Eduard', 'Emhar', 'Esci', 'Evolution',
    'First To Fight', 'Fujimi', 'GerMan', 'Gulliver', 'Hasegawa', 'HaT',
    'Hegemony', 'Heller', 'HYTTY', 'IMEX', 'Italeri', 'Legio', 'Linear-A',
    'Linear-B', 'Lucky Toys', 'LW', 'Mars', 'Matchbox', 'Metch', 'MiniArt',
    'MM', 'Model Kasten', 'MPC', 'Nexus', 'Odemars', 'Orion', 'Panzer vs Tanks',
    'Pegasus', 'Plastic Soldier', 'Pobeda', 'Preiser', 'RedBox', 'Revell',
    'Strelets', 'T-Model', 'Toxso', 'Tragik', 'Ultima Ratio', 'Valdemar',
    'Valiant', 'Waterloo 1815', 'Ykreol', 'Zvezda'
]
MANUFACTURERS_SORTED = sorted(MANUFACTURERS, key=len, reverse=True)

def get_manufacturer(header):
    """Возвращает производителя из названия набора по справочнику PSR."""
    for m in MANUFACTURERS_SORTED:
        if header.startswith(m + ' '):
            return m
    return header.split(' ')[0]  # запасной вариант, если производитель не в справочнике

df['Manufacturer'] = df['Header'].apply(get_manufacturer)

# Контроль: сколько строк не распознано по справочнику
unknown = ~df['Manufacturer'].isin(MANUFACTURERS)
print("Строк с производителем вне справочника:", unknown.sum())
print(df['Manufacturer'].value_counts().head(10))

Строк с производителем вне справочника: 0
Manufacturer
Strelets    448
HaT         348
Zvezda      176
RedBox      155
Mars        145
Italeri     120
Caesar      118
Preiser      94
Revell       84
Linear-A     79
Name: count, dtype: int64


## Шаг 5.3. Расчётные признаки

In [6]:
CURRENT_YEAR = 2026  # фиксируем год анализа для воспроизводимости

# --- 1. Длительность и середина исторического периода ---
# Работает и для отрицательных годов (до н.э.)
df['Period_Duration'] = df['Years_to'] - df['Years_from']
df['Mid_Year'] = (df['Years_from'] + df['Years_to']) // 2

# --- 2. Историческая эра (вспомогательное поле для навигации в BI) ---
# Основной упор анализа будет сделан на конкретные годы (Release_Year, Years_from, Years_to),
# а не на крупные эры, поскольку эры слишком агрегированы для детального анализа.
def get_era(mid_year):
    if pd.isna(mid_year):
        return None
    if mid_year <= 500:    return 'Древний мир'
    if mid_year <= 1600:   return 'Средневековье'
    if mid_year <= 1900:   return 'Новое время'
    if mid_year <= 1945:   return 'Новейшее время'
    return 'Современность'

df['Era'] = df['Mid_Year'].apply(get_era)

# --- 3. Десятилетие выпуска и давность релиза (для Recency в RFM) ---
# Эти поля — ключевые для анализа, а не Era
df['Decade'] = (df['Release_Year'] // 10) * 10
df['Years_Since_Release'] = CURRENT_YEAR - df['Release_Year']

# --- 4. Контроль распределения ---
print("=== Распределение по эрам (вспомогательное поле) ===")
print(df['Era'].value_counts())

print("\n=== Распределение по десятилетиям выпуска (основное поле) ===")
print(df['Decade'].value_counts().sort_index())

print("\n=== Примеры строк с расчётными признаками ===")
cols = ['Header', 'Years_from', 'Years_to', 'Mid_Year', 'Era', 
        'Release_Year', 'Decade', 'Years_Since_Release']
print(df[cols].head(10))

=== Распределение по эрам (вспомогательное поле) ===
Era
Новейшее время    933
Новое время       927
Древний мир       433
Средневековье     380
Современность     167
Name: count, dtype: int64

=== Распределение по десятилетиям выпуска (основное поле) ===
Decade
1950.0      2
1960.0     56
1970.0    127
1980.0     94
1990.0    136
2000.0    819
2010.0    854
2020.0    321
Name: count, dtype: int64

=== Примеры строк с расчётными признаками ===
                                              Header  Years_from  Years_to  \
0                   Accurate British Infantry (7200)        1768      1796   
1                   Accurate American Militia (7201)        1775      1780   
2                     Accurate Union Infantry (7202)        1861      1865   
3               Accurate Union Artillery Team (7204)        1861      1865   
4                     Accurate Union Pioneers (7205)        1861      1865   
5  Accurate Hundred Years War English Men-At-Arms...        1340      1450   
6  Acc

## Шаг 5.4. Перевод показателя национальности из широкого формата в длинный. Итоговая консолидация

In [7]:
# Разворачиваем 11 колонок национальностей в две: ID и Nationality
nat_cols = [c for c in df_nat_wide.columns if c.startswith('Nationality_')]
df_nat_long = (
    df_nat_wide.melt(id_vars=['ID'], value_vars=nat_cols, value_name='Nationality')
    .dropna(subset=['Nationality'])          # убираем пустые ячейки широкой таблицы
    [['ID', 'Nationality']]
)

# Итоговая консолидированная таблица: одна строка = пара "набор — национальность".
# Наборы без национальностей остаются одной строкой с Nationality = NaN.
df_long = df.merge(df_nat_long, on='ID', how='left')

print("Строк на уровне наборов:", len(df))
print("Строк на уровне 'набор—национальность':", len(df_long))
print("Наборов без национальностей:", df_long['Nationality'].isna().sum())

Строк на уровне наборов: 2840
Строк на уровне 'набор—национальность': 3937
Наборов без национальностей: 364


## Шаг 5.5. Сохранение результатов консолидации

In [8]:
# Таблица уровня набора — для расчётов по производителям, годам, рейтингам
df.to_csv('df_sets.csv', index=False, encoding='utf-8')

# Консолидированная длинная таблица — для BI и анализа ниш
df_long.to_csv('df_consolidated.csv', index=False, encoding='utf-8')

print("Файлы сохранены: df_sets.csv, df_consolidated.csv")

Файлы сохранены: df_sets.csv, df_consolidated.csv
